In [2]:
# Step 1: Import necessary libraries and load the dataset
# We use pandas to read and work with the CSV file like a table

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import joblib
import os

# Create outputs folders if they don't exist (good practice)
os.makedirs('outputs/plots', exist_ok=True)
os.makedirs('outputs/results', exist_ok=True)

# Load the dataset
# Make sure the file is placed exactly here: data/credit_risk_dataset.csv
df = pd.read_csv('./../data/credit_risk_dataset.csv')

# Show basic information about the dataset
print("=== Dataset Shape ===")
print(df.shape)                    # (rows, columns)

print("\n=== First 5 rows ===")
print(df.head())

print("\n=== Column Names ===")
print(df.columns.tolist())

print("\n=== Data Types ===")
print(df.dtypes)

print("\n=== Missing Values ===")
print(df.isnull().sum())

print("\n=== Basic Statistics ===")
print(df.describe())

# Save a quick summary for later use in report
df.info()

=== Dataset Shape ===
(32581, 12)

=== First 5 rows ===
   person_age  person_income person_home_ownership  person_emp_length  \
0          22          59000                  RENT              123.0   
1          21           9600                   OWN                5.0   
2          25           9600              MORTGAGE                1.0   
3          23          65500                  RENT                4.0   
4          24          54400                  RENT                8.0   

  loan_intent loan_grade  loan_amnt  loan_int_rate  loan_status  \
0    PERSONAL          D      35000          16.02            1   
1   EDUCATION          B       1000          11.14            0   
2     MEDICAL          C       5500          12.87            1   
3     MEDICAL          C      35000          15.23            1   
4     MEDICAL          C      35000          14.27            1   

   loan_percent_income cb_person_default_on_file  cb_person_cred_hist_length  
0                 0.59 

# Member 4: SVM (Support Vector Machine) + Feature Scaling + Model Evaluation

## Simple Explanation of SVM

**What is SVM?**  
Support Vector Machine is like drawing a straight line (or a plane in higher dimensions) that best separates two groups of data points. 

Imagine you have two types of students:
- Students who will default on loan (red points)
- Students who will NOT default (blue points)

SVM tries to draw the widest possible gap (called "margin") between these two groups so that the line is as far as possible from both groups. This makes the prediction more reliable.

**Why do we use SVM in this project?**
- It works well when data is not too messy.
- It is good at handling both numerical and categorical data (after preprocessing).
- It can find complex decision boundaries using something called "kernel trick".
- In loan default prediction, it helps us clearly separate customers who are likely to default from those who are not.

**Types of SVM we will use:**
- Linear SVM (simple straight line)
- RBF Kernel SVM (can handle more complex patterns - we will use this)

**Our Responsibility as Member 4:**
1. Apply Feature Scaling (very important for SVM)
2. Train SVM model
3. Evaluate the model using Accuracy, Precision, Recall, F1-score, Confusion Matrix
4. Compare all 4 models (Logistic Regression, Decision Tree, Random Forest, SVM)

In [3]:
# Step 3: Feature Scaling - Very Important for SVM

print("=== Step 3: Feature Scaling ===")

# Separate features (X) and target (y)
# We assume 'loan_status' is the target column (0 = no default, 1 = default)
# Change the column name ONLY if your dataset uses a different name

X = df.drop('loan_status', axis=1)   # All columns except target
y = df['loan_status']                # Target column

print("Features shape:", X.shape)
print("Target shape:", y.shape)

# Identify numerical columns for scaling
# We scale only numerical features (not categorical yet)
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print("\nNumerical columns that will be scaled:", numerical_cols)

# Apply StandardScaler (makes mean=0 and standard deviation=1)
scaler = StandardScaler()

# Fit and transform only numerical columns
X_scaled = X.copy()
X_scaled[numerical_cols] = scaler.fit_transform(X[numerical_cols])

print("\n=== After Scaling (first 5 rows of numerical features) ===")
print(X_scaled[numerical_cols].head())

# Save the scaler for later use (important for deployment)
joblib.dump(scaler, 'outputs/results/scaler.pkl')
print("\nScaler saved to: outputs/results/scaler.pkl")

=== Step 3: Feature Scaling ===
Features shape: (32581, 11)
Target shape: (32581,)

Numerical columns that will be scaled: ['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length']

=== After Scaling (first 5 rows of numerical features) ===
   person_age  person_income  person_emp_length  loan_amnt  loan_int_rate  \
0   -0.903374      -0.114143          28.535538   4.019404       1.545580   
1   -1.060904      -0.911147           0.050769  -1.358650       0.039595   
2   -0.430783      -0.911147          -0.914816  -0.646849       0.573479   
3   -0.745843      -0.009274          -0.190627   4.019404       1.301784   
4   -0.588313      -0.188358           0.774958   4.019404       1.005524   

   loan_percent_income  cb_person_cred_hist_length  
0             3.931411                   -0.691554  
1            -0.657458                   -0.938167  
2             3.744110                   -0.691554  
3     

In [4]:
# Step 4: Handling Categorical Variables + Final Data Preparation

print("=== Step 4: Handling Categorical Variables ===")

# Identify categorical columns (object type = text)
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

print("Categorical columns:", categorical_cols)

# Apply One-Hot Encoding (converts categories to 0 and 1 columns)
X_final = pd.get_dummies(X_scaled, columns=categorical_cols, drop_first=True)

print("\nShape after encoding:", X_final.shape)
print("New column names (first 10):", X_final.columns.tolist()[:10])

# Now we have fully numerical data ready for SVM
print("\n=== Final Data Ready for Modeling ===")
print("Features (X_final) shape:", X_final.shape)
print("Target (y) shape:", y.shape)

# Split the data into training and testing sets
# This is very important - we train on one part and test on unseen data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, 
    test_size=0.2,      # 20% for testing
    random_state=42,    # for reproducible results
    stratify=y          # keep same proportion of default/non-default
)

print("\nTraining set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

=== Step 4: Handling Categorical Variables ===
Categorical columns: ['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']

Shape after encoding: (32581, 22)
New column names (first 10): ['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'person_home_ownership_OTHER', 'person_home_ownership_OWN', 'person_home_ownership_RENT']

=== Final Data Ready for Modeling ===
Features (X_final) shape: (32581, 22)
Target (y) shape: (32581,)

Training set shape: (26064, 22)
Testing set shape: (6517, 22)


/var/folders/g8/zy5ymq393x9bz1czp761p5740000gn/T/ipykernel_66392/2376414707.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
